In [1]:
import pandas as pd
import re
from datetime import datetime

In [12]:
# Downloaded data as of 09/22/2026
dwdf = pd.read_csv('911-calls-download.csv')

new_col_names = [col.lower().replace(' ', '_') for col in dwdf.columns]
dwdf.columns = new_col_names
print(dwdf.shape)
print(dwdf.columns)

dwdf.call_time = pd.to_datetime(dwdf.call_time)

print("911 calls min date:", dwdf.call_time.min())
dwdf.code_description[dwdf.code_description.str.startswith('SHOT')].value_counts()

(2934264, 26)
Index(['incident_entry_id', 'incident_id', 'nearest_intersection',
       'call_source', 'code_description', 'category', 'priority', 'call_group',
       'nature_code', 'call_time', 'intake_time', 'dispatch_time',
       'travel_time', 'on_scene_time', 'total_response_time', 'total_time',
       'zip_code', 'precinct', 'scout_car_area', 'neighborhood_name',
       'council_district', 'longitude', 'latitude', 'esri_oid', 'x', 'y'],
      dtype='str')
911 calls min date: 2016-09-20 09:36:12


code_description
SHOTS FIRED IP                   64385
SHOTS J/H, EVIDENCE, REPT         5770
SHOT SPOTTER                      5410
SHOTSPOTTER                       2640
SHOT OR STABBED DELTA              916
SHOT STABBED UNK OR OBV DEATH      188
Name: count, dtype: int64

In [14]:
ogdf = pd.read_csv('/Users/dkinsman/Downloads/ShotSpotterDiD/data/clean_updated_gunshots.csv')
ogdf_ids = ogdf.incident_id

In [26]:
print(len(set(ogdf_ids).difference(dwdf.incident_id)))
print(len(set(ogdf_ids).union(dwdf.incident_id)))
print('Example of mismatch', list(set(ogdf_ids).difference(dwdf.incident_id))[:2])

7921
2942185
Example of mismatch [202315300870, 202321100808]


In [ ]:
201921704548
201921704548

In [5]:
wcap = datetime.strptime('2022/09/30', '%Y/%m/%d')
dwdf[dwdf['call_time']<=wcap]

,incident_entry_id,incident_id,nearest_intersection,call_source,code_description,category,priority,call_group,nature_code,call_time,...,zip_code,precinct,scout_car_area,neighborhood_name,council_district,longitude,latitude,esri_oid,x,y
728009,222713963,202227103963,Santa Maria St & Prairie St,PHONE,VERIFIED ALR / PERSON W/O CODE,VERALRM,2,12PCT,805020,2022-09-29 00:57:22,...,48221,12.0,1208,Bagley,2.0,-83.145228,42.418942,728010,NaN,NaN
728010,222711694,202227101694,Gilchrist St & W 7 Mile Rd,W911,AUTO X OR PED X - INJURIES,ACCINJ,1,8PCT,811010,2022-09-28 16:02:21,...,48235,8.0,804,Greenfield,2.0,-83.212776,42.430206,728011,NaN,NaN
728012,222721526,202227201526,Fenkell St & Coyle St,W911,ASSAULT AND BATTERY IP-JH,AB IP/JH,2,2PCT,347020,2022-09-29 14:53:34,...,48227,2.0,201,Belmont,2.0,-83.193817,42.401674,728013,NaN,NaN
728013,222713953,202227103953,Casino St & Lanark St,PHONE,SHOT SPOTTER,SHOTSPT,1,9PCT,SST342,2022-09-29 00:55:46,...,48224,9.0,907,Moross-Morang,4.0,-82.934918,42.422264,728014,NaN,NaN
728014,222711693,202227101693,Carrie St & Milbank St,PHONE,RUBBISH LITTERING I/P,LITTER,3,11PCT,873035,2022-09-28 16:02:13,...,48234,11.0,1104,Sherwood,3.0,-83.030698,42.443522,728015,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2356133,162640035,201626400035,Chalfonte St & Stansbury St,E911,ASSAULT AND BATTERY,AB IP/JH,2,2PCT,347020,2016-09-20 09:57:23,...,48227,2.0,202,Bethune Community,7.0,-83.182187,42.399810,2356134,NaN,NaN
2356134,162640032,201626400032,Russell St & E Canfield St,PHONE,UNKNOWN PROBLEM,UNKPROB,2,7PCT,843020,2016-09-20 09:55:59,...,48207,7.0,706,Forest Park,5.0,-83.048119,42.357846,2356135,NaN,NaN
2356135,162640024,201626400024,Robson St & Joy Rd,PHONE,DISTURBANCE,DISTURB,3,2PCT,825030,2016-09-20 09:50:29,...,48228,2.0,207,We Care Community,7.0,-83.191588,42.358254,2356136,NaN,NaN
2356136,162640019,201626400019,Ardmore St & W 7 Mile Rd,E911,AUTO X UNK INJ / IMPAIRED,ACCUNK,2,12PCT,811020,2016-09-20 09:46:32,...,48235,12.0,1201,Tri-Point,2.0,-83.185197,42.430911,2356137,NaN,NaN


In [7]:
dwdf['call_source'].unique()
dwdf['code_description'].unique()
dwdf['category'].unique()
dwdf['call group'].unique()

KeyError: 'call group'

In [ ]:
"""FUNCTION FROM PULL_CLEAN_CALLS.PY"""
def clean_911(df:pd.DataFrame, keep_shape = False):
    # Rename columns
    print("Original df shape:", df.shape)
    df.columns = ['incident_id', 'agency', 'incident_address', 'zip_code', 'priority', 'callcode', 'calldescription', 'category', 
                  'call_timestamp','precinct_sca', 'respondingunit', 'officerinitiated', 'intaketime','dispatchtime', 'traveltime', 
                  'totalresponsetime', 'time_on_scene', 'totaltime', 'neighborhood', 'block_id', 'council_district', 'longitude', 
                  'latitude', 'shape', 'ObjectId', 'X', 'Y']

    # Drop rows w. missing values
    df.dropna(subset = ['dispatchtime', 'traveltime', 'time_on_scene', 'longitude', 'latitude'], inplace = True)
    print("After dropping NaNs in dispatchtime, traveltime, time_on_scene, longitude, or latitude:", df.shape)
    # Drop duplicate entries
    df.drop_duplicates(subset='incident_id', inplace=True)
    print("After dropping calls with duplicate incident_ids:", df.shape)
    
    # Drop miscellaneous entris
    df = df[(df.zip_code != '     ') & (df.zip_code != '0    ')]
    print("After dropping calls without a zipcode:", df.shape)
    
    df = df[df.officerinitiated == 'No']
    print("After dropping calls initiated by an officer:", df.shape)

    df = df[(df.intaketime != '0')|( df.traveltime != '0')]
    print("After dropping calls without an intake time or travel time:", df.shape)
    
    df = df[df.calldescription != 'SYSTEM TEST - SHOTSPOTTER']
    print("After dropping shotspotter test calls:", df.shape)
    
    df = df[df.totalresponsetime <= 180]
    print("After dropping calls with response times greater than 180 minutes:", df.shape)

    print("FINAL DF SHAPE:", df.shape)

    df.call_timestamp = pd.to_datetime(df.call_timestamp, unit = 'ms')
    df.incident_id = df.incident_id.astype('int64')
    df.zip_code = df.zip_code.astype('int64')
    df.priority = df.priority.astype('int64')
    df.longitude = df.longitude.astype('float64')
    df.latitude = df.latitude.astype('float64')

    time_features = ['intaketime',
       'dispatchtime', 'traveltime', 'totalresponsetime', 'time_on_scene',
       'totaltime']
    df[time_features] = df[time_features].replace(',', '', regex=True).astype('float64')

    df['sca']= [re.sub('[^0-9]','', str(x)).lstrip('0') for x in df.sca]
    df['precinct'] = [x[:1] if x[:1] != '1' else x[:2] for x in df.sca]
    df = df[df.sca != '']
    df['sca'] = [x[:2] + '1' + x[2:] if (x == '121')| (x == '111') else x for x in df.sca]
    df['sca']= [x if ((len(x)==4) | ((len(x)==3) & (x[:1] != '1'))) else (x[:1] + '0' + x[1:] if len(x)==2 else x[:2] + '0' + x[2:]) for x in df.sca]

    df['date'] = df.call_timestamp.dt.strftime('%Y-%m-%d')
    # df['month'] = df.call_timestamp.dt.strftime('%Y-%m')
    # df['week'] = df.call_timestamp.dt.strftime('%Y-%U')
    df['year'] = df.call_timestamp.dt.strftime('%Y')

    # df.drop('precinct_sca', axis = 1, inplace = True)

    if keep_shape == True:
        df.to_csv('clean_updated_gunshots.csv', index = False)
    else:
        df.drop(['shape', 'ObjectId', 'X', 'Y'], axis = 1, inplace = True)
        df.to_csv('clean_updated_gunshots.csv', index = False)

    print("Sanity check after saving:", df.shape)
    return df   

In [ ]:
clean_911(dwdf)

Original df shape: (2934264, 26)


ValueError: Length mismatch: Expected axis has 26 elements, new values have 27 elements